# Lab 1 — Perceptron, Adaline, and the XOR Problem

Implementation and comparison of single-layer classifiers on bipolar logic (targets ±1). NumPy only.

## 1. Bipolar logic datasets

Truth tables for AND, OR, and XOR encoded with inputs and labels in {−1, +1}.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# AND: output +1 only when both inputs are +1
D_and = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]], dtype=np.float64)
t_and = np.array([1, -1, -1, -1], dtype=np.float64)

# OR: output +1 when at least one input is +1
D_or = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]], dtype=np.float64)
t_or = np.array([1, 1, 1, -1], dtype=np.float64)

# XOR: output +1 when inputs differ
D_xor = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]], dtype=np.float64)
t_xor = np.array([-1, 1, 1, -1], dtype=np.float64)

for name, D, t in [("AND", D_and, t_and), ("OR", D_or, t_or), ("XOR", D_xor, t_xor)]:
    print(name, "inputs:", D.tolist(), "labels:", t.tolist())

AND inputs: [[1.0, 1.0], [1.0, -1.0], [-1.0, 1.0], [-1.0, -1.0]] labels: [1.0, -1.0, -1.0, -1.0]
OR inputs: [[1.0, 1.0], [1.0, -1.0], [-1.0, 1.0], [-1.0, -1.0]] labels: [1.0, 1.0, 1.0, -1.0]
XOR inputs: [[1.0, 1.0], [1.0, -1.0], [-1.0, 1.0], [-1.0, -1.0]] labels: [-1.0, 1.0, 1.0, -1.0]


## 2. Visualising linear separability

Scatter (x₁, x₂) with colour by label. AND and OR admit a separating line; XOR does not.

In [1]:
def scatter_by_class(ax, D, t, title):
    mask_pos = t == 1
    mask_neg = t == -1
    ax.scatter(D[mask_pos, 0], D[mask_pos, 1], c="#059669", marker="^", s=100, label="+1", zorder=2)
    ax.scatter(D[mask_neg, 0], D[mask_neg, 1], c="#b91c1c", marker="v", s=100, label="-1", zorder=2)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
    ax.grid(True, alpha=0.25)
    ax.set_aspect("equal")

_, axs = plt.subplots(1, 3, figsize=(11, 4))
scatter_by_class(axs[0], D_and, t_and, "AND")
scatter_by_class(axs[1], D_or, t_or, "OR")
scatter_by_class(axs[2], D_xor, t_xor, "XOR")
plt.tight_layout()
plt.show()

## 3. Rosenblatt Perceptron

Weight update only on error: **Δw = α · t · x** (target *t*). Binary threshold: out = +1 if net ≥ 0 else −1.

In [1]:
class RosenblattPerceptron:
    def __init__(self, dim, lr=0.1, max_iter=100):
        self.dim = dim
        self.weights = np.zeros(dim + 1)
        self.lr = lr
        self.max_iter = max_iter

    @staticmethod
    def _threshold(z):
        return np.where(z >= 0, 1.0, -1.0)

    def train(self, D, t):
        X = np.column_stack([np.ones(len(D)), D])
        errors_per_epoch = []
        for _ in range(self.max_iter):
            n_wrong = 0
            for i in range(X.shape[0]):
                z = X[i] @ self.weights
                y = self._threshold(z)
                if y != t[i]:
                    self.weights += self.lr * t[i] * X[i]
                    n_wrong += 1
            errors_per_epoch.append(n_wrong)
            if n_wrong == 0:
                break
        return np.array(errors_per_epoch)

    def classify(self, D):
        X = np.column_stack([np.ones(len(D)), D])
        return self._threshold(X @ self.weights)

In [1]:
rp_and = RosenblattPerceptron(2, lr=0.1, max_iter=100)
err_and = rp_and.train(D_and, t_and)
rp_or = RosenblattPerceptron(2, lr=0.1, max_iter=100)
err_or = rp_or.train(D_or, t_or)
print("Perceptron AND: epochs to convergence =", len(err_and))
print("Perceptron OR:  epochs to convergence =", len(err_or))
print("AND predictions:", rp_and.classify(D_and))
print("OR  predictions:", rp_or.classify(D_or))

Perceptron AND: epochs to convergence = 3
Perceptron OR:  epochs to convergence = 3
AND predictions: [ 1. -1. -1. -1.]
OR  predictions: [ 1.  1.  1. -1.]


In [1]:
def draw_separator(ax, w):
    if np.abs(w[2]) < 1e-9:
        return
    x1 = np.linspace(-1.5, 1.5, 100)
    x2 = -(w[0] + w[1] * x1) / w[2]
    ax.plot(x1, x2, "k-", lw=2, label="Separating line")

_, axs = plt.subplots(1, 2, figsize=(9, 4))
scatter_by_class(axs[0], D_and, t_and, "Perceptron — AND")
draw_separator(axs[0], rp_and.weights)
axs[0].legend()
scatter_by_class(axs[1], D_or, t_or, "Perceptron — OR")
draw_separator(axs[1], rp_or.weights)
axs[1].legend()
plt.tight_layout()
plt.show()

## 4. Adaline (LMS / delta rule)

Update uses **net input** (no threshold in the rule): **Δw = α (t − y_in) x**. Train with linear output; at test time use threshold: +1 if y_in ≥ 0 else −1.

In [1]:
class LMSNeuron:
    def __init__(self, dim, lr=0.1, max_iter=100, seed=None):
        self.dim = dim
        self.lr = lr
        self.max_iter = max_iter
        if seed is None:
            self.weights = np.zeros(dim + 1)
        else:
            self.weights = np.random.RandomState(seed).randn(dim + 1) * 0.01

    def train(self, D, t):
        X = np.column_stack([np.ones(len(D)), D])
        mse_log = []
        for _ in range(self.max_iter):
            for j in range(X.shape[0]):
                y_in = X[j] @ self.weights
                self.weights += self.lr * (t[j] - y_in) * X[j]
            y_in_all = X @ self.weights
            mse_log.append(np.mean((t - y_in_all) ** 2))
            if mse_log[-1] < 1e-6:
                break
        return np.array(mse_log)

    def classify(self, D):
        X = np.column_stack([np.ones(len(D)), D])
        return np.where(X @ self.weights >= 0, 1.0, -1.0)

In [1]:
lms_and = LMSNeuron(2, lr=0.1, max_iter=500)
mse_and = lms_and.train(D_and, t_and)
lms_or = LMSNeuron(2, lr=0.1, max_iter=500)
mse_or = lms_or.train(D_or, t_or)
print("Adaline AND: final MSE =", mse_and[-1], ", epochs =", len(mse_and))
print("Adaline OR:  final MSE =", mse_or[-1], ", epochs =", len(mse_or))
print("AND predictions:", lms_and.classify(D_and))
print("OR  predictions:", lms_or.classify(D_or))

Adaline AND: final MSE = 0.25432525951557095 , epochs = 500
Adaline OR:  final MSE = 0.25432525951557095 , epochs = 500
AND predictions: [ 1. -1. -1. -1.]
OR  predictions: [ 1.  1.  1. -1.]


In [1]:
_, axs = plt.subplots(1, 2, figsize=(9, 4))
axs[0].plot(mse_and, color="#059669")
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("MSE")
axs[0].set_title("Adaline on AND")
axs[0].grid(True, alpha=0.25)
axs[1].plot(mse_or, color="#0d9488")
axs[1].set_xlabel("Epoch")
axs[1].set_ylabel("MSE")
axs[1].set_title("Adaline on OR")
axs[1].grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 5. XOR: single-layer limitation

Training both models on XOR; neither can achieve zero error.

In [1]:
rp_xor = RosenblattPerceptron(2, lr=0.1, max_iter=500)
err_xor = rp_xor.train(D_xor, t_xor)
print("Perceptron on XOR: errors in last 5 epochs:", err_xor[-5:].tolist())
print("Does not converge (errors stay positive).")

Perceptron on XOR: errors in last 5 epochs: [4, 4, 4, 4, 4]
Does not converge (errors stay positive).


In [1]:
plt.figure(figsize=(5, 3.5))
plt.plot(err_xor, color="#b91c1c")
plt.xlabel("Epoch")
plt.ylabel("Misclassifications")
plt.title("Perceptron on XOR — no convergence")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [1]:
lms_xor = LMSNeuron(2, lr=0.1, max_iter=500, seed=99)
mse_xor = lms_xor.train(D_xor, t_xor)
w = lms_xor.weights
print("Adaline on XOR: final weights (bias, w1, w2) =", w.tolist())
print("Decision line: {:.4f} + {:.4f}*x1 + {:.4f}*x2 = 0".format(w[0], w[1], w[2]))
print("Final MSE =", mse_xor[-1])

Adaline on XOR: final weights (bias, w1, w2) = [-4.163336342344337e-17, 0.05882352941176468, 0.11764705882352944]
Decision line: -0.0000 + 0.0588*x1 + 0.1176*x2 = 0
Final MSE = 1.0173010380622838


In [1]:
plt.figure(figsize=(5, 3.5))
plt.plot(mse_xor, color="#7c3aed")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Adaline on XOR — MSE does not go to zero")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
scatter_by_class(ax, D_xor, t_xor, "XOR and Adaline decision line")
draw_separator(ax, lms_xor.weights)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Two-layer network for XOR (fixed weights)

2–2–1 layout: H1 ≈ AND, H2 ≈ OR; output = step(−1 − H1 + H2) gives XOR.

In [1]:
def step(z):
    return np.where(z >= 0, 1.0, -1.0)

# Hidden: H1 = AND (weights to H1), H2 = OR (weights to H2)
W_hidden = np.array([[-1.5, 1, 1], [0.5, 1, 1]])
# Output: step(bias + w1*H1 + w2*H2) with [bias, w1, w2] = [-1, -1, 1]
w_out = np.array([-1.0, -1.0, 1.0])

def two_layer_xor(D):
    ones = np.ones((len(D), 1))
    X = np.column_stack([ones, D])
    H = step(X @ W_hidden.T)
    H_bias = np.column_stack([ones, H])
    return step(H_bias @ w_out)

pred = two_layer_xor(D_xor)
print("Two-layer XOR output:", pred)
print("Targets:            ", t_xor)
print("All correct:", np.allclose(pred, t_xor))

Two-layer XOR output: [-1.  1.  1. -1.]
Targets:             [-1.  1.  1. -1.]
All correct: True
